# Reference response training with LoRA

For the final training, I fine-tune Qwen2.5-7B on each of the 8 selected data groups, keeping the training objective and adapter settings the same across all methods.

The objective is to have the model produce a response similar to its safe counterpart's response when receiving an injection prompt. I use LoRA to train small added matrices within selected model projections, while the base model's weights stay fixed.

- [The training experiment](#The-training-experiment)
- [A training pair example](#A-training-pair-example)
- [How the reference-response loss works](#How-the-reference-response-loss-works)
- [Final training settings](#Final-training-settings)


## The training experiment

Each method provides 256 FIPI training pairs, with the injection prompt as input and the base model's response to its safe counterpart as the reference answer.

The dictionary variable `config` contains these shared settings and the 8 selection methods. The reference answers are model-generated responses, and their task correctness was not systematically verified.


In [ ]:
# inspect injection inputs, safe reference answers, and the LoRA training settings

import json
from pathlib import Path

import pandas as pd
from IPython.display import display

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "scripts/curation/configs/config_reference_groups.json").exists())
config = json.loads((root / "scripts/curation/configs/config_reference_groups.json").read_text())

# read the approved reference-response run separately from the earlier separation-loss configurations
fields = ["model", "methods", "training_pairs", "validation_pairs", "testing_pairs", "objective", "reference_review"]
display(pd.Series({field: config[field] for field in fields}, name="configuration").to_frame())


## A training pair example

The variable `method` chooses one selected data group, and the dictionary variable `pair` contains its first prepared training example.

I decode the injection input beside its safe reference answer to show exactly what the training compares, with the dictionary field `task_response_ids` containing the complete reference token sequence, including the end token.


In [ ]:
from transformers import AutoTokenizer

# the method chooses the training data while the model and loss stay the same
method = "random"
with (root / config["data"] / "training" / f"{method}.jsonl").open() as stream:
    pair = json.loads(next(stream))

# decoding reads the tokenizer only, without loading model weights or generating text
tokenizer = AutoTokenizer.from_pretrained(root / config["model"], local_files_only=True)
pd.set_option("display.max_colwidth", None)
display(pd.Series({
    "pair ID": pair["pair_id"],
    "injection input": tokenizer.decode(pair["prefix_ids"]["injection"], skip_special_tokens=False),
    "safe reference answer": tokenizer.decode(pair["task_response_ids"], skip_special_tokens=False),
    "reference tokens including end token": len(pair["task_response_ids"]),
}).to_frame("training example"))


## How the reference-response loss works

I use the average negative log probability of the safe reference tokens as the training loss, meaning the model receives a larger penalty when it assigns a lower probability to the response that I want it to produce.

$$L = -\frac{1}{T}\sum_{t=1}^{T}\log P_{\theta}(y_t \mid x_{\mathrm{injection}}, y_{<t})$$

$x_{\mathrm{injection}}$: the prompt containing the injected instruction  
$y_t$: the reference token at position $t$  
$y_{<t}$: the reference tokens before that token  
$T$: the number of reference tokens, including the end token  
$\theta$: the model parameters, with only its LoRA adapter parameters updated during training

The loss takes the negative sum of the log probabilities assigned to each reference token, given the injection prompt AND the earlier reference tokens, then divides by the number of reference tokens.

Supplying the earlier reference tokens is called teacher forcing, and assigning a higher probability to the desired next token lowers its loss. The function `task_response_loss()` below implements this calculation over the complete reference response.


In [ ]:
import ast

# inspect how the loss uses the probability assigned to each safe reference token
source = (root / "scripts/curation/training/reference_loss.py").read_text()
# display the function that calculates the reference-response loss
loss_function = next(node for node in ast.parse(source).body if isinstance(node, ast.FunctionDef) and node.name == "task_response_loss")
print(ast.get_source_segment(source, loss_function))


## Final training settings

I train each adapter for 5 epochs, using LoRA on the query, value, MLP expansion and MLP reduction projections in layers 10 through 19. The parameter `rank` sets the inner size of each LoRA update, while `alpha` scales the update by alpha divided by rank.

After all 5 epochs, I choose the checkpoint with the lowest reference-response validation loss for testing, with an earlier epoch selected if losses tie. The table below lists the shared training settings.


In [ ]:
# show the shared adapter settings and the validation rule for checkpoint selection
fields = ["layers", "target_modules", "rank", "alpha", "dropout", "epochs", "pairs_per_update", "gradient_accumulation", "optimizer", "learning_rate", "weight_decay", "warmup_updates", "schedule", "total_updates", "gradient_clip", "max_input_tokens", "max_task_response_tokens", "evaluation_max_new_tokens", "compute_dtype", "seed", "validation", "checkpoint"]
display(pd.Series({field: config[field] for field in fields}, name="setting").to_frame())


Above shows the chosen configuration shared by all 8 selection groups, keeping the model and training settings fixed so I can compare the effect of selecting different training data.

The final objective uses supervised reference-response training, while the earlier ODILE-based experiment used hidden state separation losses.
